# Week 4 — Multilingual Ablation Study, MBERT
**Project**: Quantum Kernels — Multilingual (full ablations, all 6 languages)

**For each of 6 languages:** the standard three ablations
1. Feature maps — Angle vs IQP 1L (reused from Week 3) vs IQP 2L
2. Qubit count — 4 vs 8
3. C sweep (N=200)

**Resume-safe:** each ablation kernel matrix is saved immediately and reused on
re-run, so the ~48 hr total survives session deaths. Completed languages/maps
are skipped automatically.

**Inputs per language:** `multiling_mbert_<lang>_embeddings.npz`,
`multiling_mbert_<lang>_kernel_matrices.npz` (Week 3 IQP-1L), `multiling_mbert_<lang>_baseline_results.pkl`
**Metric:** macro F1 | **Runtime:** ~8 hrs per language-set... wait, ~4 hrs x 6 = ~24 hrs per encoder

In [ ]:
!pip install pennylane pennylane-lightning scikit-learn numpy --quiet
import pennylane as qml, numpy as np, time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__}")
dev4 = qml.device('lightning.qubit', wires=4)
dev8 = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage

In [ ]:
TAG = 'multiling_mbert'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")
def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"; fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname); return dp if os.path.exists(dp) else fname
print("Helpers ready (Drive + download).")

---
## Step 1: Feature maps + config

In [ ]:
LANGUAGES = ['english', 'spanish', 'german', 'arabic', 'chinese', 'hindi']
def mf1(yt, yp): return float(f1_score(yt, yp, average='macro'))

def angle_map(x, wires):
    for i, w in enumerate(wires): qml.RY(x[i], wires=w)
def iqp_1layer(x, wires):
    n=len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i,w in enumerate(wires): qml.RZ(x[i],wires=w)
    for i in range(n-1):
        qml.CNOT(wires=[wires[i],wires[i+1]]); qml.RZ(x[i]*x[i+1],wires=wires[i+1]); qml.CNOT(wires=[wires[i],wires[i+1]])
def iqp_2layer(x, wires):
    for _ in range(2): iqp_1layer(x, wires)
def make_kernel(fmap, device, n_wires):
    @qml.qnode(device)
    def circ(x1,x2):
        fmap(x1,wires=range(n_wires)); qml.adjoint(fmap)(x2,wires=range(n_wires)); return qml.probs(wires=range(n_wires))
    return lambda a,b: float(circ(a,b)[0])
k_angle=make_kernel(angle_map,dev8,8); k_iqp1=make_kernel(iqp_1layer,dev8,8)
k_iqp2=make_kernel(iqp_2layer,dev8,8); k_iqp1_4q=make_kernel(iqp_1layer,dev4,4)
print("Feature maps ready.")

---
## Step 2: Per-language ablations

Loops over all 6 languages. For each language, runs the three ablations with
immediate saves + reuse. A saved matrix is never recomputed.

In [ ]:
ablation_summary = {}

for lang in LANGUAGES:
    ltag = f'{TAG}_{lang}'
    emb_path = _find(f'{ltag}_embeddings.npz')
    wk3_path = _find(f'{ltag}_kernel_matrices.npz')
    bl_path  = _find(f'{ltag}_baseline_results.pkl')
    if not (os.path.exists(emb_path) and os.path.exists(wk3_path) and os.path.exists(bl_path)):
        print(f"[skip] {lang}: missing Week 2/3 files")
        continue

    print(f"\n{'#'*56}\n# Language: {lang}\n{'#'*56}")
    data = np.load(emb_path)
    X_tr = data['X_tr_subset'].astype(np.float64); y_tr = data['y_tr_subset']
    X_te = data['X_te_subset'].astype(np.float64); y_te = data['y_te_subset']
    wk3 = np.load(wk3_path); K_wk3_train = wk3['K_train_quantum']; K_wk3_test = wk3['K_test_quantum']
    with open(bl_path,'rb') as f: baselines = pickle.load(f)
    linear_f1 = baselines.get('Linear',{}).get('f1',0)

    def load_or_compute(name, kfn, Xa, Xb):
        path = _find(f'{ltag}_kernel_{name}_n1000.npz')
        if os.path.exists(path):
            print(f"    reuse {name}"); m=np.load(path); return m['K_train'], m['K_test']
        print(f"    computing {name} (~83 min)...")
        t0=time.time()
        Ktr=qml.kernels.kernel_matrix(Xa,Xa,kfn); Kte=qml.kernels.kernel_matrix(Xb,Xa,kfn)
        print(f"      done {(time.time()-t0)/60:.1f} min — saving")
        checkpoint(f'{ltag}_kernel_{name}_n1000.npz', lambda p,a=Ktr,b=Kte: np.savez(p,K_train=a,K_test=b))
        return Ktr, Kte

    # --- Ablation 1: feature maps ---
    fm={}
    y_w3=SVC(kernel='precomputed',C=1.0).fit(K_wk3_train,y_tr).predict(K_wk3_test)
    fm['IQP 1L']={'accuracy':float(accuracy_score(y_te,y_w3)),'f1':mf1(y_te,y_w3),'reused':True}
    for nm,kfn,key in [('Angle',k_angle,'angle'),('IQP 2L',k_iqp2,'iqp2l')]:
        Ktr,Kte=load_or_compute(key,kfn,X_tr,X_te)
        yp=SVC(kernel='precomputed',C=1.0).fit(Ktr,y_tr).predict(Kte)
        fm[nm]={'accuracy':float(accuracy_score(y_te,yp)),'f1':mf1(y_te,yp)}

    # --- Ablation 2: qubit count (4q) ---
    K4tr,K4te=load_or_compute('iqp1l_4q',k_iqp1_4q,X_tr[:,:4],X_te[:,:4])
    yp4=SVC(kernel='precomputed',C=1.0).fit(K4tr,y_tr).predict(K4te)
    qub={'8':{'accuracy':fm['IQP 1L']['accuracy'],'f1':fm['IQP 1L']['f1']},
         '4':{'accuracy':float(accuracy_score(y_te,yp4)),'f1':mf1(y_te,yp4)}}

    # --- Ablation 3: C sweep at N=200 ---
    Kc_tr=qml.kernels.kernel_matrix(X_tr[:200],X_tr[:200],k_iqp1); Kc_te=qml.kernels.kernel_matrix(X_te,X_tr[:200],k_iqp1)
    csweep={}
    for C in [0.1,0.5,1.0,5.0,10.0]:
        yp=SVC(kernel='precomputed',C=C).fit(Kc_tr,y_tr[:200]).predict(Kc_te)
        csweep[str(C)]={'accuracy':float(accuracy_score(y_te,yp)),'f1':mf1(y_te,yp)}
    best_C=max(csweep,key=lambda c: csweep[c]['f1'])

    ablation_summary[lang]={'feature_map':fm,'qubit':qub,'c_sweep':csweep,'best_C':best_C,'linear_f1':linear_f1}
    print(f"  {lang}: FM IQP2L f1={fm['IQP 2L']['f1']:.4f}  4q f1={qub['4']['f1']:.4f}  bestC={best_C}")
    checkpoint(f'{ltag}_results_week4.json', lambda p,r=ablation_summary[lang]: json.dump(r,open(p,'w'),indent=2))
    # progress
    checkpoint(f'{TAG}_week4_progress.json', lambda p,r=ablation_summary: json.dump(r,open(p,'w'),indent=2))

print(f"\nAblations done for: {list(ablation_summary.keys())}")

---
## Step 3: Cross-language ablation figure

Shows the feature-map effect across all languages — does the same circuit win
everywhere? Consistency across languages = robust general finding.

In [ ]:
if ablation_summary:
    langs = list(ablation_summary.keys())
    fmaps = ['Angle','IQP 1L','IQP 2L']
    fig, ax = plt.subplots(figsize=(12,5))
    x = np.arange(len(langs)); w = 0.25
    cmap = {'Angle':'#4D9DE0','IQP 1L':'#E15554','IQP 2L':'#3BB273'}
    for i,fm_name in enumerate(fmaps):
        vals=[ablation_summary[l]['feature_map'][fm_name]['f1'] for l in langs]
        ax.bar(x+(i-1)*w, vals, w, label=fm_name, color=cmap[fm_name], edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(langs, rotation=15)
    ax.set_ylim(0.2,1.0); ax.set_ylabel('Macro F1')
    ax.set_title(f'Feature map ablation across languages (MBERT)')
    ax.legend(); plt.tight_layout(); plt.show()
    save_figure(fig, f'{TAG}_fig_ablation_crosslang')
    checkpoint(f'{TAG}_week4_all.json', lambda p: json.dump(ablation_summary,open(p,'w'),indent=2))
    print(f"\n=== WEEK 4 DONE — {TAG} ({len(langs)} languages) ===")
else:
    print("No languages completed.")

---
## Week 4 done — Multilingual MBERT

### Saved per language:
- `multiling_mbert_<lang>_kernel_angle_n1000.npz`, `_iqp2l_`, `_iqp1l_4q_`
- `multiling_mbert_<lang>_results_week4.json`
### Combined:
- `multiling_mbert_fig_ablation_crosslang.png/.pdf` — feature map effect across all languages
- `multiling_mbert_week4_all.json`

### Resume:
If a session dies, just re-run — finished kernel matrices are reused, finished
languages skip fast. The progress file `multiling_mbert_week4_progress.json` tracks state.

### Run both encoders for the complete multilingual ablation set.